# 01 — Exploratory Data Analysis: Lending Club Credit Risk

This notebook explores the Gold-layer feature set produced by the medallion pipeline.  
**Goal:** Understand feature distributions, class balance, correlations, and potential modeling challenges before training.

**Data lineage:**  
`CSV.gz → Bronze (raw Parquet) → Silver (cleaned, typed, validated) → Gold (engineered features, train/val/test split)`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load Gold data
GOLD_DIR = Path("../data/gold")

with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)

train = pd.read_parquet(GOLD_DIR / "features_train.parquet")
val = pd.read_parquet(GOLD_DIR / "features_val.parquet")
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")

feature_cols = meta["feature_columns"]

print(f"Train: {len(train):,} rows | Val: {len(val):,} rows | Test: {len(test):,} rows")
print(f"Features: {len(feature_cols)}")
print(f"Split method: {meta['split_method']}")
train.head()

## 1. Target Variable — Class Balance & Temporal Drift

In [ ]:
# Default rates across splits (temporal drift)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: default rate per split
splits = ['Train (<2016)', 'Val (2016-H1 2017)', 'Test (>=Jul 2017)']
rates = [train['default'].mean(), val['default'].mean(), test['default'].mean()]
counts = [len(train), len(val), len(test)]

ax = axes[0]
bars = ax.bar(splits, rates, color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black', alpha=0.8)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{rate:.1%}', ha='center', fontweight='bold')
ax.set_ylabel('Default Rate')
ax.set_title('Default Rate by Time Period')
ax.set_ylim(0, 0.35)

# Pie chart: class balance in training set
ax = axes[1]
default_counts = train['default'].value_counts()
ax.pie(default_counts, labels=['Non-Default', 'Default'], autopct='%1.1f%%',
       colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.05))
ax.set_title(f'Training Set Class Balance (n={len(train):,})')

plt.tight_layout()
plt.show()

print(f"Default rate increases over time: {rates[0]:.1%} → {rates[1]:.1%} → {rates[2]:.1%}")
print("This temporal drift is expected and is what our monitoring agents will detect.")

## 2. Feature Distributions — Numeric Features

In [ ]:
# Key numeric features — distribution by default status
key_features = ['fico_score', 'int_rate', 'dti', 'annual_inc', 'loan_amnt', 
                'revol_util', 'loan_to_income', 'credit_history_months']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    for label, color, name in [(0, '#2ecc71', 'Non-Default'), (1, '#e74c3c', 'Default')]:
        data = train[train['default'] == label][col].dropna()
        # Clip outliers for visualization
        q99 = data.quantile(0.99)
        data = data[data <= q99]
        ax.hist(data, bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Default Status (Training Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
# Correlation matrix — focus on correlation with target
corr = train[feature_cols + ['default']].corr()

# Top correlations with default
target_corr = corr['default'].drop('default').sort_values(key=abs, ascending=False)
print("Top 15 features by |correlation with default|:")
print(target_corr.head(15).to_string())
print()

# Heatmap of top 15 features + target
top_features = target_corr.head(15).index.tolist() + ['default']
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(train[top_features].corr(), annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Correlation Matrix — Top 15 Features by Target Correlation', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Default Rate by Grade & FICO Band

In [ ]:
# Default rate by grade and FICO band
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By grade
grade_map_inv = {1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E', 6: 'F', 7: 'G'}
grade_stats = train.groupby('grade_numeric').agg(
    default_rate=('default', 'mean'), count=('default', 'size')
).reset_index()
grade_stats['grade'] = grade_stats['grade_numeric'].map(grade_map_inv)

ax = axes[0]
bars = ax.bar(grade_stats['grade'], grade_stats['default_rate'], 
              color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 7)), edgecolor='black')
for bar, rate, cnt in zip(bars, grade_stats['default_rate'], grade_stats['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{rate:.1%}\n(n={cnt:,})', ha='center', fontsize=9)
ax.set_ylabel('Default Rate')
ax.set_title('Default Rate by Lending Club Grade')
ax.set_ylim(0, 0.55)

# By FICO band
train['fico_band'] = pd.cut(train['fico_score'], bins=[600, 650, 700, 750, 800, 850], 
                             labels=['600-650', '650-700', '700-750', '750-800', '800-850'])
fico_stats = train.groupby('fico_band', observed=True).agg(
    default_rate=('default', 'mean'), count=('default', 'size')
).reset_index()

ax = axes[1]
bars = ax.bar(fico_stats['fico_band'].astype(str), fico_stats['default_rate'],
              color=plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(fico_stats))), edgecolor='black')
for bar, rate, cnt in zip(bars, fico_stats['default_rate'], fico_stats['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{rate:.1%}\n(n={cnt:,})', ha='center', fontsize=9)
ax.set_ylabel('Default Rate')
ax.set_title('Default Rate by FICO Band')
ax.set_ylim(0, 0.35)

plt.tight_layout()
plt.show()

## 5. Summary Statistics & Data Quality

In [ ]:
# Summary stats
print("=== Training Set Summary ===")
print(train[feature_cols].describe().T.to_string())
print()

# Null check (should be zero after Silver processing)
nulls = train[feature_cols].isnull().sum()
if nulls.sum() == 0:
    print("Data quality: No nulls in any feature column.")
else:
    print("Columns with nulls:")
    print(nulls[nulls > 0])

# Check for infinite values
inf_check = np.isinf(train[feature_cols].select_dtypes(include=[np.number])).sum()
if inf_check.sum() == 0:
    print("Data quality: No infinite values.")
else:
    print("Columns with inf values:")
    print(inf_check[inf_check > 0])